# Mid-Training, Annealing, and Continual Pretraining

> Pretraining is done, and the model is ready to use? Not quite. Experience from 2024 shows that the last 5% of training steps hold a critical operation that can push the model to the next level.
>
> This section covers three things: the WSD scheduler that makes training resumable at any point, annealing that maximizes the value of computation in the final stage, and continual pretraining that turns a general model into a domain expert.

The traditional Cosine learning rate scheduler has a fundamental flaw: the total number of training steps must be fixed in advance. If you want to continue training partway through, the learning rate curve becomes discontinuous and the model may collapse. The WSD (Warmup-Stable-Decay) scheduler splits training into three phases -- warmup, stable, and decay -- where the Stable phase keeps the learning rate constant, allowing training to be extended at any time.

Annealing borrows from metallurgy: gradually reducing the learning rate in the final stage of training lets model parameters move from rough convergence toward fine-grained alignment. Continual Pretraining (CPT) extends this idea: training a general model further on domain-specific data while using data mixing and LoRA freezing to prevent catastrophic forgetting.

The core of WSD, annealing, and CPT is the learning rate scheduling strategy. The WSD scheduler is the foundation for all three, so we will implement it first and understand the role of each phase.

## 1. The Problem

Suppose you trained a 7B model with a Cosine learning rate scheduler on 1T tokens. After training, you realize that with 500B more tokens, the loss could probably drop further.

**But here is the problem:**

The Cosine scheduler's learning rate curve rises first and then falls, like an inverted bowl. Before using it, you must decide the total number of training steps T in advance -- the scheduler plans when to start decreasing the learning rate based on T. When training reaches T, the learning rate has already dropped to nearly 0.

Now you want to continue training? The learning rate has already bottomed out -- parameter updates are nearly zero, so training further is effectively useless. The model has completed its preset steps, but the data has not been fully used. The cost of continuing is resetting the learning rate -- but the Cosine curve has already dropped to its floor and cannot rise again.

This is the structural flaw of the Cosine scheduler: it assumes you know when training should end before it begins. In practice, "when to stop" is often not determined in advance -- you want to keep training when loss is still decreasing, or stop early when results are poor. Cosine does not support this flexibility.

In 2024, the MiniCPM team proposed a solution: the WSD scheduler.

## 2. The WSD Scheduler: Training That Can Resume Anytime

### 2.1 Three Phases: Warmup, Stable, Decay

WSD divides training into three segments:

```
         Warmup    │     Stable (constant)    │  Decay (anneal)
  eta_max ────┐    │                          │
              /     │                          │
             /      │                          │
            /       │                          │    \
           /        │                          │     \
  0 ─────┘          │                          │      ──
       first 5%     │     middle 80-85%        │  last 10-15%
```

| Phase | LR change | Duration | Purpose |
|:------|:----------|:---------|:--------|
| Warmup | 0 to max | first 5% of steps | Let the model "warm up" -- avoid large steps from the start |
| Stable | **constant** | middle 80-85% | **Full-speed training**, can stop anytime |
| Decay | max to 0 | last 10-15% | "Annealing" -- fine-grained adjustment |

The core innovation is the **Stable phase**: LR stays constant, so the model is always at "full power" during training.
This means:
- You can start a Decay from any checkpoint in the Stable phase
- You can extend training indefinitely -- train as long as you want, then do a final Decay
- You do not need to know the total training volume in advance

In [ ]:
import numpy as np

np.random.seed(42)
# === Cosine vs WSD: LR comparison at key positions ===
print("=== Cosine vs WSD: LR comparison at key positions ===")
print()

T = 1000
warmup = 50
stable_end = 850
eta_max = 0.01

def cosine_lr(t):
    """Cosine: starts decreasing right after warmup"""
    if t < warmup:
        return eta_max * t / warmup
    progress = (t - warmup) / (T - warmup)
    return eta_max * 0.5 * (1 + np.cos(np.pi * progress))

def wsd_lr(t):
    """WSD: warmup -> stable (constant) -> decay"""
    if t < warmup:
        return eta_max * t / warmup
    elif t < stable_end:
        return eta_max  # constant!
    else:
        progress = (t - stable_end) / (T - stable_end)
        return eta_max * (0.001 ** progress)

print(f"{'Step':>6s}  {'Phase':>8s}  {'Cosine LR':>12s}  {'WSD LR':>12s}  {'Difference'}")
print("-" * 60)

for t in [0, 50, 200, 500, 700, 850, 900, 950, 999]:
    if t < warmup:
        phase = "Warmup"
    elif t < stable_end:
        phase = "Stable"
    else:
        phase = "Decay"

    cos = cosine_lr(t)
    wsd = wsd_lr(t)
    diff = f"WSD is {wsd/cos:.1f}x Cosine" if cos > 0 else "--"
    print(f"{t:>6d}  {phase:>8s}  {cos:>12.6f}  {wsd:>12.6f}  {diff}")

print()
print("Key observations:")
print("  Step 500: Cosine has dropped to 0.004, WSD is still at full power 0.01")
print("  Step 700: Cosine is down to 0.001, WSD is still at full power")
print("  -> Cosine training gets slower and slower, WSD stays at full speed")
print()
print("Conclusion: Want to continue training? WSD is ready anytime. Cosine? LR is already at the floor.")

In [ ]:
import matplotlib.pyplot as plt
# === Visualization: Cosine vs WSD ===
import numpy as np

steps = np.arange(T)
cos_lrs = [cosine_lr(t) for t in steps]
wsd_lrs = [wsd_lr(t) for t in steps]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: normal training
ax = axes[0]
ax.plot(steps, cos_lrs, 'b-', lw=1.5, label='Cosine')
ax.plot(steps, wsd_lrs, 'r-', lw=1.5, label='WSD')
ax.axvspan(0, warmup, alpha=0.1, color='green', label='Warmup')
ax.axvspan(warmup, stable_end, alpha=0.05, color='blue', label='Stable')
ax.axvspan(stable_end, T, alpha=0.1, color='red', label='Decay')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine vs WSD')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: continuation scenario
ax = axes[1]
ext = np.arange(600, 1200)
cos_ext = [cosine_lr(min(t, T-1)) for t in ext]  # Cosine has already dropped to 0
wsd_ext = [wsd_lr(t) if t < T else eta_max * (0.001 ** ((t - stable_end) / 200)) for t in ext]
ax.plot(ext, cos_ext, 'b-', lw=1.5, label='Cosine (continue training)')
ax.plot(ext, wsd_ext, 'r-', lw=1.5, label='WSD (continue training)')
ax.axvline(x=600, color='green', lw=1.5, label='Continuation start')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Continuation: cosine is near zero')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Left: WSD maintains full speed for 85% of the time")
print("Right: When continuing training, Cosine has almost no LR left, WSD still runs at full speed")

### 2.2 Overtraining + WSD = 2024 Standard Recipe

Recall from Part 9: LLaMA 3 8B used D/N = 1875x data (Chinchilla suggests 20x is sufficient).
This is called "overtraining" -- feeding small models more data saves money at inference time.

**But Cosine does not support unlimited overtraining.** You have to decide the training duration in advance.

WSD solves this problem:

```
Traditional (Cosine):
  Determine data amount -> Determine total steps T -> Set Cosine by T -> Train and done

Modern (WSD):
  No upper limit, WSD stable keeps training -> Stop when you want -> Final decay
  -> Overtraining goes from "theory" to "operational engineering practice"
```

| Model | Params N | Data D | D/N | Scheduler |
|:------|:---------|:-------|:----|:----------|
| Chinchilla (2022) | 70B | 1.4T | 20x | Cosine |
| MiniCPM (2024) | 2.4B | ~1T | ~417x | **WSD** |
| LLaMA 3 8B (2024) | 8B | 15T | 1875x | Cosine (but the industry is shifting to WSD) |

## 3. Annealing

The Decay phase of WSD is also called "annealing." The name comes from metallurgy: metal heated to high temperature is slowly cooled, and during the cooling process the internal atoms rearrange into a more ordered, stronger crystal structure.

Annealing during training has a similar effect. The process of reducing the learning rate from high to low lets model parameters transition from "exploring in large steps" to "fine-grained convergence."

### 3.1 Why Annealing Works

The Stable phase has a large learning rate. The benefit of a large learning rate is a wide exploration range -- parameters take large steps on the loss surface and are less likely to get trapped in local optima. But the cost is randomness in each step: parameters oscillate around the optimal solution without a chance to "settle down."

In the Decay phase, the learning rate gradually decreases and parameter updates become smaller. By this point the model has already found the right general direction -- the large number of updates during the Stable phase has given it a rough picture of the loss surface. With the correct direction known, smaller steps are actually more precise: each step moves toward the minimum rather than bouncing around it.

In other words: the Stable phase determines "which direction to go," and the Decay phase accomplishes "arriving at the destination precisely."

### 3.2 The Role of High-Quality Data During Annealing

The annealing phase involves more than just changing the learning rate -- the training data also switches. This is like a sprint review in the week before an exam: instead of doing lots of practice tests, you review your error notebook and key notes.

In practice, the training data during the annealing phase is mixed with 30-50% high-quality data (Wikipedia, curated Q&A, textbook-level text), so the model encounters the best learning materials when its "steps are getting smaller." If these high-quality data were fed during the Stable phase, the model's steps would be too large and the fine-grained signals from good data could be overwhelmed by the noise of coarse data.

In [ ]:
# === Annealing phase Loss simulation ===
import numpy as np

print("=== Annealing: Loss drops sharply in the final stage ===")
print()

np.random.seed(42)
total_steps = 1000
stable_end = 850

steps = np.arange(total_steps)
loss = np.zeros(total_steps)

# Stable phase: loss decreases slowly
for t in range(stable_end):
    progress = (t + 1) / stable_end
    loss[t] = 3.0 * (progress ** (-0.05)) + np.random.normal(0, 0.02)

# Decay phase: loss drops faster (annealing effect)
final_stable = loss[stable_end - 1]
for t in range(stable_end, total_steps):
    progress = (t - stable_end) / (total_steps - stable_end)
    drop = 0.15 * (1 - np.exp(-progress * 5))  # additional drop
    loss[t] = final_stable - drop + np.random.normal(0, 0.01)

# Calculate cost-effectiveness
stable_avg = np.mean(loss[800:850])
decay_avg = np.mean(loss[-50:])
improvement = stable_avg - decay_avg

stable_per_step = (4.0 - stable_avg) / stable_end
decay_per_step = improvement / (total_steps - stable_end)

print(f"Late Stable loss: {stable_avg:.4f}")
print(f"Late Decay loss:  {decay_avg:.4f}")
print(f"Improvement from annealing: {improvement:.4f} ({improvement/stable_avg*100:.1f}%)")
print()
print(f"Stable improvement per step: {stable_per_step:.6f}")
print(f"Decay improvement per step:  {decay_per_step:.6f}")
print(f"Decay per-step efficiency vs Stable: {decay_per_step/stable_per_step:.1f}x")
print()
print("Key: Annealing uses only 15% of steps, but per-step efficiency is actually higher!")

In [ ]:
# === Annealing visualization ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# Top: full loss curve
ax = axes[0]
ax.plot(steps, loss, 'b-', lw=0.8, alpha=0.7)
ax.axvline(x=stable_end, color='red', ls='--', lw=1.5, label=f'Decay starts (step {stable_end})')
ax.axvspan(0, 50, alpha=0.1, color='green', label='Warmup')
ax.axvspan(50, stable_end, alpha=0.05, color='blue', label='Stable')
ax.axvspan(stable_end, 1000, alpha=0.1, color='red', label='Decay annealing')
ax.set_ylabel('Loss')
ax.set_title('WSD training loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Bottom: zoom into decay
ax = axes[1]
ax.plot(steps[750:], loss[750:], 'b-', lw=1.5)
ax.axvline(x=stable_end, color='red', ls='--', lw=2, label='Decay starts')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Zoom: loss drops faster during annealing')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Notice how the descent slope in the decay region (red) becomes noticeably steeper -- this is the power of annealing.")

## 4. Mid-Training Data Strategy

The annealing phase involves more than just LR decreasing -- **data is also being switched.**

Intuition: just like "sprint review" in the week before an exam -- instead of doing practice tests (coarse data), you review your error notebook and key notes (refined data).

MiniCPM's ablation study compared three strategies:

| Strategy | Stable phase data | Decay phase data | Effect |
|:---------|:------------------|:-----------------|:-------|
| A: No switch | CC+Code+Books | Same as left (coarse data) | baseline |
| B: SFT after annealing | CC+Code+Books | Same as A, then do SFT after annealing | Better |
| C: **Mix during annealing** | CC+Code+Books | **CC+Code+Books + high-quality data** | **Best** |

Why is strategy C the best?
- Early in annealing, LR is still relatively large -> the model has "capacity" to absorb new data patterns
- Late in annealing, LR is very small -> fine-grained convergence locks the new patterns into parameters
- This uses the LR decay process to simultaneously accomplish "learning new content + consolidating old knowledge"

In [ ]:
# === Decay phase data mix calculation ===
print("=== Annealing Phase Data Mix Schemes ===")
print()

# Scenario: 2.4B model, total 1.2T tokens
# Stable: 1T (coarse data)  Decay: 200B (coarse + refined mix)
decay_total = 200  # B tokens

schemes = [
    ("No switch",        100,  0,  0,  0),
    ("Conservative mix",  70, 10, 10, 10),
    ("Balanced mix",      50, 20, 15, 15),
    ("Aggressive mix",    30, 30, 20, 20),
]

print(f"{'Scheme':<20s} {'Coarse':>8s} {'Wiki':>8s} {'SFT':>8s} {'Instruct':>8s} {'Assessment'}")
print("-" * 70)
for name, coarse, wiki, sft, inst in schemes:
    c = decay_total * coarse / 100
    w = decay_total * wiki / 100
    s = decay_total * sft / 100
    i = decay_total * inst / 100
    print(f"{name:<20s} {c:>6.0f}B  {w:>6.0f}B  {s:>6.0f}B  {i:>6.0f}B  ", end="")
    if coarse == 100:
        print("baseline")
    elif coarse >= 50:
        print("recommended")
    else:
        print("domain learned, but high forgetting risk")

print()
print("Golden rules:")
print("  1. Coarse data (CC+Code) must not be dropped -> prevents forgetting pretrained knowledge")
print("  2. High-quality data should be 30-50% -> lets annealing truly show its fine-tuning effect")
print("  3. High-quality data over 50% biases the model -> forgets diverse knowledge")

In [ ]:
# === Data mix visualization ===
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

names = ['No switch', 'Conservative', 'Balanced', 'Aggressive']
coarse = [100, 70, 50, 30]
quality = [0, 30, 50, 70]  # total high-quality data

x = np.arange(len(names))
ax.bar(x, coarse, label='Coarse data (CC+Code)', color='#95a5a6', alpha=0.8)
ax.bar(x, quality, bottom=coarse, label='High-quality data (Wiki+SFT+instructions)', color='#2ecc71', alpha=0.8)
ax.set_ylabel('Share (%)')
ax.set_title('Data mix during decay')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend()
ax.axhline(y=50, color='red', ls='--', alpha=0.5, label='50% reference line')

# Mark recommended range
ax.axvspan(0.5, 2.5, alpha=0.05, color='green')
ax.text(1.5, 55, 'Recommended range', ha='center', fontsize=10, color='green')

plt.tight_layout()
plt.show()

## 5. Continual Pretraining (CPT)

Annealing and data switching are strategies "within pretraining" -- they do not change the model's positioning. But there is a bigger scenario:

> **You already have a general LLM (e.g., LLaMA 3), and now you want it to learn specialized knowledge in medicine, law, or finance.**

This is called **Continual Pretraining (CPT)**.

### 5.1 The Core Challenge: Catastrophic Forgetting

The biggest risk with CPT is catastrophic forgetting. During pretraining on general data, the model learns a wide range of knowledge -- grammar, common sense, reasoning, multilingual ability. When you continue training only on domain data, the model gradually "forgets" general knowledge because new gradient updates overwrite old knowledge.

An analogy: someone who has studied English for three years will make progress in French if they study only French for the next three months, but their English will deteriorate. If they review English every day alongside French, both languages can be maintained.

CPT needs to do exactly the same thing -- learn the new domain while preserving general capabilities.

### 5.2 The Golden Rule of Data Mixing

In [ ]:
# === CPT data strategy calculation ===
print("=== CPT Data Mix: Domain Data vs General Data ===")
print()

cpt_total = 50  # B tokens

strategies = [
    ("Pure domain",     100,   0, "Severe forgetting: model can only write medical records"),
    ("Domain-heavy",     70,  30, "General ability partially degrades"),
    ("Balanced mix",     50,  50, "Recommended: best of both worlds"),
    ("Conservative mix", 30,  70, "Insufficient domain learning"),
]

print(f"Total CPT data: {cpt_total}B tokens")
print()
print(f"{'Strategy':<20s} {'Domain':>8s} {'General':>8s} {'Assessment'}")
print("-" * 60)
for name, domain, general, note in strategies:
    d = cpt_total * domain / 100
    g = cpt_total * general / 100
    print(f"{name:<20s} {d:>6.0f}B  {g:>6.0f}B  {note}")

print()
print("Golden rule: domain data : general data = 1:1 to 2:1")
print("The role of general data = an 'anchor' that prevents the model from forgetting what it already knows")

In [ ]:
# === Forgetting vs Learning: sweet spot visualization ===
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

domain_pct = np.array([0, 30, 50, 70, 100])
forget_risk = np.array([0, 10, 25, 60, 90])  # forgetting risk
domain_learn = np.array([0, 40, 75, 90, 95])  # domain learning effect

ax2 = ax.twinx()
ax.plot(domain_pct, forget_risk, 'r-o', lw=2, ms=8, label='Forgetting risk')
ax2.plot(domain_pct, domain_learn, 'b--s', lw=2, ms=8, label='Domain learning')

ax.axvspan(40, 70, alpha=0.1, color='green', label='Recommended range')
ax.axvline(x=60, color='green', ls='--', alpha=0.7, label='Best ~60%')

ax.set_xlabel('Domain data share (%)')
ax.set_ylabel('Forgetting risk (%)', color='red')
ax2.set_ylabel('Domain learning effect (%)', color='blue')
ax.set_title('CPT sweet spot: 50-70% domain data')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='center right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Red rising = more forgetting, Blue rising = better domain learning")
print("The green overlap region is the sweet spot -- the best of both worlds.")

### 5.3 LoRA for CPT: Natural Forgetting Prevention + Memory Savings

Two paths for CPT:

| Method | Trainable params | Forgetting prevention | Memory (7B) |
|:-------|:-----------------|:----------------------|:------------|
| Full CPT | All 7B | Poor (all params changing) | ~120 GB |
| **LoRA CPT** | ~21M (0.3%) | **Good (original weights frozen)** | **~15 GB** |

LoRA naturally prevents forgetting: original weights W remain completely unchanged, only the small side branch AB is updated.
It is like adding a small extension on top of the original foundation -- the foundation stays intact.

In [ ]:
# === CPT memory comparison ===
print("=== CPT Memory Comparison (7B model) ===")
print()
print(f"{'Method':<20s} {'Base':>8s} {'Trainable':>10s} {'Optimizer':>10s} {'Total':>8s}")
print("-" * 60)

# Full CPT
print(f"{'Full CPT':<20s} {'14 GB':>8s} {'7000M':>10s} {'84 GB':>10s} {'~120 GB':>8s}")

# LoRA CPT
print(f"{'LoRA CPT (r=16)':<20s} {'14 GB':>8s} {'~21M':>10s} {'~1 GB':>10s} {'~15 GB':>8s}")

# QLoRA CPT
print(f"{'QLoRA CPT (4bit)':<20s} {'3.5 GB':>8s} {'~21M':>10s} {'~1 GB':>10s} {'~6 GB':>8s}")

print()
print("QLoRA only needs 6GB -> a single RTX 3060 can do domain CPT on a 7B model!")
print()
print("Core advantages of LoRA for CPT:")
print("  1. Original weights frozen -> foundation intact -> general knowledge not lost")
print("  2. Only train side branch AB -> extremely memory-efficient")
print("  3. After training, merge -> zero inference overhead (same speed as a normal model)")
print()
print("Note: If CPT data volume is very large (>100B), LoRA's low rank may not be sufficient.")
print("In that case, consider a larger r (e.g., 64 or 128), or use full CPT directly.")

## 6. The Big Picture: The Complete Training Pipeline from Pretraining to CPT

Let us connect everything learned in this Part into a single pipeline:

```
+------------------------------------------------------------------+
|                 The Complete LLM Training Lifecycle               |
+------------------------------------------------------------------+
|                                                                   |
|  1. Pretraining                                                   |
|     Data: Common Crawl + Wiki + Code + Books (10T+ tokens)        |
|     Scheduler: WSD (Stable phase, LR constant)                    |
|     |                                                             |
|     v                                                             |
|                                                                   |
|  2. Mid-Training Annealing                                        |
|     Data: Mix in 30-50% high-quality data (Wiki/SFT/instructions)  |
|     Scheduler: WSD Decay phase, LR decreasing                     |
|     |                                                             |
|     v                                                             |
|                                                                   |
|  3. Continual Pretraining (CPT) -- optional                       |
|     Scenario: Adapt a general model to a specific domain          |
|     Data: Domain 50-70% + General 30-50%                          |
|     Method: LoRA/QLoRA (memory savings + forgetting prevention)   |
|     |                                                             |
|     v                                                             |
|                                                                   |
|  4. SFT + RLHF (covered in a later Part)                          |
|     Teach the model "how to answer questions"                     |
|                                                                   |
+------------------------------------------------------------------+
```

**Each step builds on top of the previous one -- nothing is done from scratch.**

## Summary

Confirm that you understand these points:

- 1. The limitation of the Cosine scheduler: training steps must be fixed in advance, cannot be resumed
- 2. WSD three phases: Warmup -> Stable (constant) -> Decay (annealing)
- 3. WSD's core advantage: Stable phase LR is constant -> can stop or resume at any time
- 4. Why annealing works: direction already determined + smaller steps are more precise + high-quality data guidance
- 5. Mid-Training data strategy: mixing in 30-50% high-quality data during annealing gives the best results
- 6. CPT's core challenge: catastrophic forgetting -- learning new knowledge causes old knowledge to be lost
- 7. CPT data golden rule: domain 50-70% + general 30-50%, general data acts as the "anchor"
- 8. LoRA for CPT: freezing original weights = foundation intact + memory reduced to 1/8
- 9. Complete training lifecycle: Pretraining -> Annealing -> CPT (optional) -> SFT -> RLHF

**One-sentence summary**: Pretraining is not the endpoint, but the starting point. WSD makes training resumable at any time, annealing maximizes the effect of the last 10% of computation, and CPT turns a general model into a domain expert -- together, these three form the standard training recipe of 2024.

## Exercises

**Exercise 1: Design a domain data mixture**

Assume you are adapting a general model to medical QA. Design a CPT data mixture containing general text, medical textbooks, QA data, and safe-refusal data. Explain the role of each data type.

**Exercise 2: Observe forgetting risk**

Using a toy vocabulary, build a "general corpus" and a "domain corpus". Continue training once with only domain data, then again with 20% general replay data. Compare whether general-token loss rises more easily without replay.

**Exercise 3: Write a training-log checklist**

List metrics to check every fixed number of steps during CPT: training loss, validation loss, domain-task score, general-task score, repetition rate, safety examples, and so on. Explain what each abnormal signal might mean.